# Semantic Similarity Benchmarking

Benchmark pair extraction, metric inspection, and weight tuning live here.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from gprofiler import GProfiler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Benchmarks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
BENCHMARK_DIR = PROJECT_ROOT / "Benchmarks"
BENCHMARK_DIR.mkdir(exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from crossenrich.standardization import standardize_results_frame
from crossenrich.semantic import build_semantic_similarity_matrix

gene_list = [
    "TP53", "MDM2", "CDKN1A", "BAX", "BBC3", "PMAIP1", "GADD45A", "GADD45B",
    "SESN1", "SESN2", "RRM2B", "DRAM1", "FAS", "TNFRSF10B", "APAF1",
    "CASP3", "CASP9", "CYCS", "PTEN", "ATM", "ATR", "BRCA1", "BRCA2",
]

gp = GProfiler()
results = pd.DataFrame(
    gp.profile(gene_list, organism="hsapiens", no_evidences=False)
)
frame = standardize_results_frame(results)
semantic_similarity = build_semantic_similarity_matrix(
    frame.copy(),
    cross_source_only=False,
)


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35152.73it/s]
BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
mask = np.triu(np.ones(semantic_similarity.shape, dtype=bool), k=1)
top_pairs = (semantic_similarity.where(mask).stack().reset_index())


top_pairs.columns = ["left_index", "right_index", "similarity"]

top_pairs = top_pairs.sort_values("similarity", ascending=False)

display(top_pairs.head(20))


,left_index,right_index,similarity
28479,67,72,1.000000
48848,123,141,1.000000
25714,60,65,1.000000
12878,29,32,1.000000
98485,343,388,1.000000
57866,151,185,0.973993
85410,260,261,0.964789
9432,21,46,0.963636
99145,349,379,0.962968
15858,36,37,0.950872


In [3]:
top_pairs["left_name"] = top_pairs["left_index"].apply(lambda i: frame.iloc[i]["name"])
top_pairs["right_name"] = top_pairs["right_index"].apply(lambda i: frame.iloc[i]["name"])

top_pairs["left_source"] = top_pairs["left_index"].apply(lambda i: frame.iloc[i]["canonical_source"])
top_pairs["right_source"] = top_pairs["right_index"].apply(lambda i: frame.iloc[i]["canonical_source"])

top_pairs["left_id"] = top_pairs["left_index"].apply(lambda i: frame.iloc[i]["native"])
top_pairs["right_id"] = top_pairs["right_index"].apply(lambda i: frame.iloc[i]["native"])

top_pairs = top_pairs.sort_values("similarity", ascending=False).reset_index(drop=True)


display(top_pairs.head(20))


,left_index,right_index,similarity,left_name,right_name,left_source,right_source,left_id,right_id
0,67,72,1.000000,Melanoma,Melanoma,KEGG,WP,KEGG:05218,WP:WP4685
1,123,141,1.000000,Cell cycle,Cell cycle,WP,KEGG,WP:WP179,KEGG:04110
2,60,65,1.000000,Endometrial cancer,Endometrial cancer,KEGG,WP,KEGG:05213,WP:WP4155
3,29,32,1.000000,Small cell lung cancer,Small cell lung cancer,KEGG,WP,KEGG:05222,WP:WP4658
4,343,388,1.000000,Bladder cancer,Bladder cancer,KEGG,WP,KEGG:05219,WP:WP2828
5,151,185,0.973993,positive regulation of release of cytochrome c...,regulation of release of cytochrome c from mit...,GO:BP,GO:BP,GO:0090200,GO:0090199
6,260,261,0.964789,positive regulation of telomerase catalytic co...,regulation of telomerase catalytic core comple...,GO:BP,GO:BP,GO:1904884,GO:1904882
7,21,46,0.963636,Apoptosis,Apoptosis,WP,REAC,WP:WP254,REAC:R-HSA-109581
8,349,379,0.962968,cellular response to actinomycin D,response to actinomycin D,GO:BP,GO:BP,GO:0072717,GO:0072716
9,36,37,0.950872,Pathways in cancer,Cancer pathways,KEGG,WP,KEGG:05200,WP:WP5434


Sample results for metric testing.

In [4]:
review_df = top_pairs.copy().reset_index(drop=True)

review_df["pair_id"] = review_df.index + 1

review_df["left_name"] = review_df["left_index"].apply(lambda i: frame.iloc[i]["name"])
review_df["right_name"] = review_df["right_index"].apply(lambda i: frame.iloc[i]["name"])

review_df["left_source"] = review_df["left_index"].apply(lambda i: frame.iloc[i]["canonical_source"])
review_df["right_source"] = review_df["right_index"].apply(lambda i: frame.iloc[i]["canonical_source"])

review_df["left_id"] = review_df["left_index"].apply(lambda i: frame.iloc[i]["native"])
review_df["right_id"] = review_df["right_index"].apply(lambda i: frame.iloc[i]["native"])

review_df["expected_relation"] = ""
review_df["expected_similarity"] = ""
review_df["notes"] = ""

review_df.head(10)


,left_index,right_index,similarity,left_name,right_name,left_source,right_source,left_id,right_id,pair_id,expected_relation,expected_similarity,notes
0,67,72,1.000000,Melanoma,Melanoma,KEGG,WP,KEGG:05218,WP:WP4685,1,,,
1,123,141,1.000000,Cell cycle,Cell cycle,WP,KEGG,WP:WP179,KEGG:04110,2,,,
2,60,65,1.000000,Endometrial cancer,Endometrial cancer,KEGG,WP,KEGG:05213,WP:WP4155,3,,,
3,29,32,1.000000,Small cell lung cancer,Small cell lung cancer,KEGG,WP,KEGG:05222,WP:WP4658,4,,,
4,343,388,1.000000,Bladder cancer,Bladder cancer,KEGG,WP,KEGG:05219,WP:WP2828,5,,,
5,151,185,0.973993,positive regulation of release of cytochrome c...,regulation of release of cytochrome c from mit...,GO:BP,GO:BP,GO:0090200,GO:0090199,6,,,
6,260,261,0.964789,positive regulation of telomerase catalytic co...,regulation of telomerase catalytic core comple...,GO:BP,GO:BP,GO:1904884,GO:1904882,7,,,
7,21,46,0.963636,Apoptosis,Apoptosis,WP,REAC,WP:WP254,REAC:R-HSA-109581,8,,,
8,349,379,0.962968,cellular response to actinomycin D,response to actinomycin D,GO:BP,GO:BP,GO:0072717,GO:0072716,9,,,
9,36,37,0.950872,Pathways in cancer,Cancer pathways,KEGG,WP,KEGG:05200,WP:WP5434,10,,,


In [5]:
cross_df = review_df.copy()

clear_positive_pool = cross_df[cross_df["similarity"] > 0.75]
middle_pool = cross_df[
    (cross_df["similarity"] >= 0.4) &
    (cross_df["similarity"] <= 0.6)
]
clear_negative_pool = cross_df[cross_df["similarity"] < 0.2]

top_sample = clear_positive_pool.sample(min(20, len(clear_positive_pool)), random_state=42)
middle_sample = middle_pool.sample(min(30, len(middle_pool)), random_state=42)
low_sample = clear_negative_pool.sample(min(20, len(clear_negative_pool)), random_state=42)

top_sample = top_sample.sort_values("similarity", ascending=False)
middle_sample = middle_sample.sort_values("similarity", ascending=False)
low_sample = low_sample.sort_values("similarity", ascending=False)

benchmark_df = pd.concat(
    [top_sample, middle_sample, low_sample],
    ignore_index=True
).drop_duplicates(subset=["left_index", "right_index"]).reset_index(drop=True)

benchmark_df["pair_id"] = benchmark_df.index + 1

benchmark_df[
    [
        "pair_id",
        "left_index",
        "right_index",
        "left_source",
        "left_id",
        "left_name",
        "right_source",
        "right_id",
        "right_name",
        "similarity",
        "expected_relation",
        "expected_similarity",
        "notes",
    ]
].to_csv(BENCHMARK_DIR / "semantic_similarity_benchmark.csv", index=False)


In [6]:
top_sample.head(20)


,left_index,right_index,similarity,left_name,right_name,left_source,right_source,left_id,right_id,pair_id,expected_relation,expected_similarity,notes
5,151,185,0.973993,positive regulation of release of cytochrome c...,regulation of release of cytochrome c from mit...,GO:BP,GO:BP,GO:0090200,GO:0090199,6,,,
9,36,37,0.950872,Pathways in cancer,Cancer pathways,KEGG,WP,KEGG:05200,WP:WP5434,10,,,
33,252,311,0.894231,positive regulation of lymphocyte apoptotic pr...,positive regulation of leukocyte apoptotic pro...,GO:BP,GO:BP,GO:0070230,GO:2000108,34,,,
42,8,25,0.883906,cellular response to abiotic stimulus,response to abiotic stimulus,GO:BP,GO:BP,GO:0071214,GO:0009628,43,,,
46,199,283,0.878235,Regulation of TP53 Activity through Phosphoryl...,Regulation of TP53 Activity,REAC,REAC,REAC:R-HSA-6804756,REAC:R-HSA-5633007,47,,,
75,61,109,0.851163,response to UV,cellular response to UV,GO:BP,GO:BP,GO:0009411,GO:0034644,76,,,
140,171,203,0.815429,cellular response to starvation,cellular response to glucose starvation,GO:BP,GO:BP,GO:0009267,GO:0042149,141,,,
143,242,280,0.814723,T cell apoptotic process,lymphocyte apoptotic process,GO:BP,GO:BP,GO:0070231,GO:0070227,144,,,
154,252,380,0.807393,positive regulation of lymphocyte apoptotic pr...,positive regulation of thymocyte apoptotic pro...,GO:BP,GO:BP,GO:0070230,GO:0070245,155,,,
157,13,126,0.806407,cellular response to stress,cellular response to stimulus,GO:BP,GO:BP,GO:0033554,GO:0051716,158,,,


In [7]:
# Put two frame indices here
left_index = 25
right_index = 116

from crossenrich.semantic import (
    _build_embedding_input,
    _geometric_containment,
    _jaccard,
    _semantic_similarity,
    _trigram_jaccard,
    compute_semantic_similarity,
    model,
)

left = frame.loc[left_index]
right = frame.loc[right_index]

left_semantic_text = _build_embedding_input(left["name"], left["description"])
right_semantic_text = _build_embedding_input(right["name"], right["description"])
texts = [left_semantic_text, right_semantic_text]
embeddings = model.encode(texts)
embeddings_cache = dict(zip(texts, embeddings))

pd.Series(
    {
        "token_overlap": _geometric_containment(left["term_tokens"], right["term_tokens"]),
        "gene_list_overlap": _jaccard(left["intersection_genes"], right["intersection_genes"]),
        "lexical_similarity": _trigram_jaccard(left["description"], right["description"]),
        "semantic_similarity": _semantic_similarity(
            left_semantic_text,
            right_semantic_text,
            embeddings_cache,
        ),
        "final_similarity": compute_semantic_similarity(left, right, embeddings_cache),
    }
)


token_overlap          0.000000
gene_list_overlap      0.809524
lexical_similarity     0.180723
semantic_similarity    0.906368
final_similarity       0.577510
dtype: float64